In [0]:
import json
import re
import shutil
from datetime import datetime, timezone
from pathlib import Path
from pyspark.sql import functions as F

In [0]:

ENTITIES = {
    "users": {"filename": "users.jsonl"},
    "products": {"filename": "products.jsonl"},
    "carts": {"filename": "carts.jsonl"},
    "orders": {"filename": "orders.jsonl"},
}

BASE_PATH = "/Workspace/Users/arfaashaikh101@gmail.com/retail_dev_dbx/"
LATEST_BATCH_ID = "batch_001"
FILENAME = ENTITIES.get('users')['filename']
LANDING_FILE = Path(BASE_PATH) / "landing" / LATEST_BATCH_ID / FILENAME

In [0]:
source_schema = """
    id BIGINT,
    firstName STRING,
    lastName STRING,
    email STRING,
    address STRUCT<country: STRING>
"""

df_users_raw = (
    spark.read
    .format("json")
    .schema(source_schema)
    .option("multiLine", "false")
    .option("mode", "FAILFAST")
    .load(str(LANDING_FILE))
)

In [0]:
df_users_raw_1 = df_users_raw.withColumn('country', df_users_raw.address.country) \
                             .withColumnRenamed("firstName", "first_name") \
                             .withColumnRenamed("lastName", "last_name") \
                             .withColumn("batch_id", F.lit(LATEST_BATCH_ID)) \
                             .withColumn("ingested_at", F.timestamp_seconds(F.unix_timestamp())) \
                             .drop("address")
display(df_users_raw_1)

In [0]:
target_table = "workspace.retail_dev_schema.bronze_users"

(
    df_users_raw_1.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"batch_id = '{LATEST_BATCH_ID}'")
    .saveAsTable(target_table)
)